In [17]:
#PASO 1: CARGA DEl DATASET ZONAS

import pandas as pd
import os

# Configuración de rutas
ruta_tablas = '../../datos/base_datos'
pd.set_option('display.max_columns', None)

# 1.1 Carga de la tabla maestra de Zonas (la base de las 31 filas)
df_zonas = pd.read_csv(os.path.join(ruta_tablas, 'zonas', 'ZONAS.csv'))

# 1.2 Visualización previa 
print("-" * 70)
print("📊 Previsualización de ZONAS:")
print("-" * 70)
display(df_zonas.head())

# 1.3 Filtramos las columnas necesarias
# Mantenemos ID_ZONA, NOMBRE_ZONA y AREA_KM2
df_zonas = df_zonas[['ID_ZONA', 'NOMBRE_ZONA', 'AREA_KM2']].copy()

# 1.4 Verificación
print("-" * 70)
print(f"✅ Tabla ZONAS lista (Base del Master). Filas: {df_zonas.shape[0]}")
print("-" * 70)
display(df_zonas.head())

----------------------------------------------------------------------
📊 Previsualización de ZONAS:
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,NOMBRE_DISTRITO,AREA_KM2
0,MN0101,Financial District-Battery Park City,Manhattan,19.2268
1,MN0102,Tribeca-Civic Center,Manhattan,13.5782
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,Manhattan,11.9933
3,MN0201,SoHo-Little Italy-Hudson Square,Manhattan,12.9168
4,MN0202,Greenwich Village,Manhattan,10.6005


----------------------------------------------------------------------
✅ Tabla ZONAS lista (Base del Master). Filas: 38
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2
0,MN0101,Financial District-Battery Park City,19.2268
1,MN0102,Tribeca-Civic Center,13.5782
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168
4,MN0202,Greenwich Village,10.6005


In [18]:
#PASO 2: CARGA Y SELECCIÓN DEL DATASET CENSO

# 2.1 Carga del dataset
df_censo = pd.read_csv(os.path.join(ruta_tablas, 'censo', 'CENSO.csv'))

# 2.2 Visualización previa 
print("-" * 70)
print("📊 Previsualización de CENSO:")
print("-" * 70)
display(df_censo.head(3))

# 2.3 Selección de las variables 
columnas_finales_censo = [
    'ID_ZONA', 
    'POBLACION_TOTAL', 
    'EDAD_MEDIANA', 
    'INGRESO_MEDIANO_HOGAR', 
    'TASA_EMPLEO', 
    'TAMANO_HOGAR_PROMEDIO', 
    'POBLACION_HISPANA',
    'TASA_OCUPACION'
]
df_censo_sel = df_censo[columnas_finales_censo].copy()

# 2.4 Unión al Master (Filtro por ID_ZONA)
# Aquí es donde el Master se queda con las 31 zonas válidas de Manhattan
df_master = pd.merge(df_zonas, df_censo_sel, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ CENSO integrado. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())


----------------------------------------------------------------------
📊 Previsualización de CENSO:
----------------------------------------------------------------------


,ID_ZONA,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION
0,MN0101,48608,34.3,198961.0,79.57,1.99,1639,83.10
1,MN0102,23257,36.8,185902.0,68.92,2.20,873,81.41
2,MN0201,22167,39.3,132900.0,70.97,1.78,975,82.01


----------------------------------------------------------------------
✅ CENSO integrado. Dimensiones del Master: (38, 10)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61


In [19]:
#PASO 3: CARGA Y SELECCIÓN DEL DATASET SEGURIDAD

# 3.1 Carga del dataset
df_seguridad = pd.read_csv(os.path.join(ruta_tablas, 'seguridad', 'SEGURIDAD.csv'))

# 3.2 Visualización previa
print("-" * 70)
print("📊 Previsualización de SEGURIDAD:")
print("-" * 70)
display(df_seguridad.head(15))

# 3.3 Definición de la lógica de subclasificación por entorno
def clasificar_entorno_riesgo(row):
    # Riesgo directo al local: Ocurre en comercio o es específicamente robo a establecimiento
    if row['LUGAR_DELITO'] == 'LOCAL COMERCIAL' or row['TIPO_DELITO'] == 'ROBO A ESTABLECIMIENTO':
        return 'SEG_DELITOS_PROPIEDAD'
    # Riesgo en el trayecto: Ocurre en el sistema de transporte
    elif row['LUGAR_DELITO'] == 'TRANSPORTE':
        return 'SEG_DELITOS_TRANSPORTE'
    # Riesgo de entorno general: Vía pública, edificios, etc.
    else:
        return 'SEG_DELITOS_OTROS'

# 3.4 Aplicamos la clasificación
df_seguridad['SUB_CATEGORIA'] = df_seguridad.apply(clasificar_entorno_riesgo, axis=1)

# 3.5 Agregación pivotada por ID_ZONA y SUB_CATEGORIA
df_seg_agrupado = df_seguridad.groupby(['ID_ZONA', 'SUB_CATEGORIA'])['CANTIDAD_INCIDENTES'].sum().unstack(fill_value=0).reset_index()

# 3.6 TRATAMIENTO DE ENTEROS
columnas_seg = ['SEG_DELITOS_PROPIEDAD', 'SEG_DELITOS_TRANSPORTE', 'SEG_DELITOS_OTROS']
for col in columnas_seg:
    if col in df_seg_agrupado.columns:
        df_seg_agrupado[col] = df_seg_agrupado[col].astype(int)

# 3.7 Limpieza de seguridad previa en el master (para evitar duplicidad o columnas viejas)
columnas_a_limpiar = ['TOTAL_DELITOS', 'SEG_DELITOS_PROPIEDAD', 'SEG_DELITOS_TRANSPORTE', 'SEG_DELITOS_OTROS']
for col in columnas_a_limpiar:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# 3.7 Unión al Master
df_master = pd.merge(df_master, df_seg_agrupado, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ SEGURIDAD integrada.. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())


----------------------------------------------------------------------
📊 Previsualización de SEGURIDAD:
----------------------------------------------------------------------


,ID_ZONA,TIPO_DELITO,LUGAR_DELITO,CANTIDAD_INCIDENTES
0,MN0101,ASALTO GRAVE,EDIFICIOS PUBLICOS,6
1,MN0101,ASALTO GRAVE,LOCAL COMERCIAL,7
2,MN0101,ASALTO GRAVE,OTROS,6
3,MN0101,ASALTO GRAVE,TRANSPORTE,18
4,MN0101,ASALTO GRAVE,VIA PUBLICA,23
5,MN0101,ASALTO GRAVE,ZONA RESIDENCIAL,4
6,MN0101,ROBO A ESTABLECIMIENTO,LOCAL COMERCIAL,26
7,MN0101,ROBO A ESTABLECIMIENTO,OTROS,9
8,MN0101,ROBO A ESTABLECIMIENTO,VIA PUBLICA,1
9,MN0101,ROBO A ESTABLECIMIENTO,ZONA RESIDENCIAL,9


----------------------------------------------------------------------
✅ SEGURIDAD integrada.. Dimensiones del Master: (38, 13)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0


In [20]:
#PASO 4: CARGA Y SELECCIÓN DEL DATASET MOVILIDAD

# 4.1 Carga del dataset
df_movilidad = pd.read_csv(os.path.join(ruta_tablas, 'movilidad', 'MOVILIDAD.csv'))

# 4.2 Visualización previa 
print("-" * 70)
print("📊 Previsualización de MOVILIDAD:")
print("-" * 70)
display(df_movilidad.head(3))
display(df_movilidad.info())

# 4.3 Agregación de datos por ID_ZONA
df_semana = df_movilidad[df_movilidad['TIPO_DIA'] == 'Laborable'].set_index('ID_ZONA')['VOLUMEN_TOTAL_PASAJEROS'].astype(int)
df_finde = df_movilidad[df_movilidad['TIPO_DIA'] == 'Fin de Semana'].set_index('ID_ZONA')['VOLUMEN_TOTAL_PASAJEROS'].astype(int)
df_estaciones = df_movilidad.groupby('ID_ZONA')['CANTIDAD_ESTACIONES'].max().astype(int)

#4.4 Consolidamos en un dataframe temporal con la nueva nomenclatura
df_movilidad_agrupado = pd.DataFrame({
    'MOV_DIA_SEMANA_PROMEDIO': df_semana,
    'MOV_FIN_DE_SEMANA_PROMEDIO': df_finde,
    'MOV_CANTIDAD_ESTACIONES': df_estaciones
}).reset_index()

# 4.5 Si las columnas ya existen en el master, la borramos antes de unir
columnas_movilidad = ['MOV_DIA_SEMANA_PROMEDIO', 'MOV_FIN_DE_SEMANA_PROMEDIO' , 'MOV_CANTIDAD_ESTACIONES']
for col in columnas_movilidad:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# 4.6 Unión al Master
df_master = pd.merge(df_master, df_movilidad_agrupado, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ MOVILIDAD integrada. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())


----------------------------------------------------------------------
📊 Previsualización de MOVILIDAD:
----------------------------------------------------------------------


,ID_ZONA,TIPO_DIA,VOLUMEN_TOTAL_PASAJEROS,CANTIDAD_ESTACIONES
0,MN0101,Fin de Semana,54974,10
1,MN0101,Laborable,78390,10
2,MN0102,Fin de Semana,36589,6


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ID_ZONA                  62 non-null     object
 1   TIPO_DIA                 62 non-null     object
 2   VOLUMEN_TOTAL_PASAJEROS  62 non-null     int64 
 3   CANTIDAD_ESTACIONES      62 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 2.1+ KB


None

----------------------------------------------------------------------
✅ MOVILIDAD integrada. Dimensiones del Master: (38, 16)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE,MOV_DIA_SEMANA_PROMEDIO,MOV_FIN_DE_SEMANA_PROMEDIO,MOV_CANTIDAD_ESTACIONES
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0,78390.0,54974.0,10.0
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0,58553.0,36589.0,6.0
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0,NaN,NaN,NaN
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0,51431.0,39477.0,6.0
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0,30640.0,23725.0,4.0


In [21]:
#PASO 5: CARGA Y SELECCIÓN DEL DATASET RESTAURANTES

# 5.1 Carga del dataset con el nombre exacto
df_restaurantes = pd.read_csv(os.path.join(ruta_tablas, 'competencia', 'RESTAURANTES.csv'))

# 5.2 Visualización previa 
print("-" * 70)
print("📊 Previsualización de RESTAURANTES:")
print("-" * 70)
display(df_restaurantes.head(3))

# 5.3 Transformación: Convertimos las categorías en columnas independientes
df_res_pivot = df_restaurantes.pivot_table(
    index='ID_ZONA', 
    columns='CLASIFICACION_COMPETENCIA', 
    values='CANTIDAD_LOCALES', 
    aggfunc='sum'
).reset_index()

# 5.4 Renombrado para claridad en el Master
df_res_pivot.rename(columns={
    'Directa (Rivales)': 'RES_COM_DIRECTA',
    'Indirecta (Tráfico)': 'RES_COM_INDIRECTA'
}, inplace=True)

#5.5 Si las columnas ya existen en el master, la borramos antes de unir
columnas_res = ['RES_COM_DIRECTA', 'RES_COM_INDIRECTA']
for col in columnas_res:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# 5.6 Unión al Master
df_master = pd.merge(df_master, df_res_pivot, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ RESTAURANTES integrada. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())


----------------------------------------------------------------------
📊 Previsualización de RESTAURANTES:
----------------------------------------------------------------------


,ID_ZONA,TIPO_COCINA,CLASIFICACION_COMPETENCIA,CANTIDAD_LOCALES
0,MN0101,American,Indirecta (Tráfico),160
1,MN0101,Asian/Asian Fusion,Indirecta (Tráfico),13
2,MN0101,Chicken,Indirecta (Tráfico),3


----------------------------------------------------------------------
✅ RESTAURANTES integrada. Dimensiones del Master: (38, 18)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE,MOV_DIA_SEMANA_PROMEDIO,MOV_FIN_DE_SEMANA_PROMEDIO,MOV_CANTIDAD_ESTACIONES,RES_COM_DIRECTA,RES_COM_INDIRECTA
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0,78390.0,54974.0,10.0,24.0,305.0
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0,58553.0,36589.0,6.0,11.0,150.0
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0,51431.0,39477.0,6.0,12.0,275.0
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0,30640.0,23725.0,4.0,15.0,242.0


In [22]:
#PASO 5: CARGA Y SELECCIÓN DEL DATASET LUGARES INTERES

# 6.1 Carga del dataset
df_lugares = pd.read_csv(os.path.join(ruta_tablas, 'lugares_interes', 'LUGARES_INTERES.csv'))

# 6.2 Visualización previa 
print("-" * 70)
print("📊 Previsualización de LUGARES DE INTERÉS:")
print("-" * 70)
display(df_lugares.head(15))


# 6.3 Re-clasificación de lugares de interes, según su comportamiento humano
mapa_final = {
    'Administracion Publica': 'LUG_OFICINAS_ESCUELAS',
    'Educacion': 'LUG_OFICINAS_ESCUELAS',
    'Comercial y Hoteles': 'LUG_COMERCIO_TURISMO',
    'Cultura': 'LUG_COMERCIO_TURISMO',
    'Monumentos y Plazas': 'LUG_COMERCIO_TURISMO',
    'Parques y Jardines': 'LUG_COMERCIO_TURISMO',
    'Residencial': 'LUG_RESIDENCIAL_EXCLUSIVO'  
}

# 6.4 Aplicación de la subcategoría
df_lugares['SUB_CATEGORIA'] = df_lugares['CATEGORIA_LUGAR'].map(mapa_final)

# 6.5 Agregación
df_lug_agrupado = df_lugares.groupby(['ID_ZONA', 'SUB_CATEGORIA'])['CANTIDAD_LUGARES'].sum().unstack(fill_value=0).reset_index()
       
# 6.6 Si las columnas ya existen en el master, la borramos antes de unir
columnas_lug = ['LUG_OFICINAS_ESCUELAS', 'LUG_COMERCIO_TURISMO', 'LUG_RESIDENCIAL_EXCLUSIVO']
for col in columnas_lug:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# 6.7 Unión al Master
df_master = pd.merge(df_master, df_lug_agrupado, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ LUGARES DE INTERÉS. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())


----------------------------------------------------------------------
📊 Previsualización de LUGARES DE INTERÉS:
----------------------------------------------------------------------


,ID_ZONA,CATEGORIA_LUGAR,CANTIDAD_LUGARES
0,MN0101,Administracion Publica,12
1,MN0101,Comercial y Hoteles,41
2,MN0101,Cultura,21
3,MN0101,Educacion,25
4,MN0101,Monumentos y Plazas,37
5,MN0101,Parques y Jardines,23
6,MN0101,Residencial,6
7,MN0102,Administracion Publica,26
8,MN0102,Comercial y Hoteles,3
9,MN0102,Cultura,2


----------------------------------------------------------------------
✅ LUGARES DE INTERÉS. Dimensiones del Master: (38, 21)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE,MOV_DIA_SEMANA_PROMEDIO,MOV_FIN_DE_SEMANA_PROMEDIO,MOV_CANTIDAD_ESTACIONES,RES_COM_DIRECTA,RES_COM_INDIRECTA,LUG_COMERCIO_TURISMO,LUG_OFICINAS_ESCUELAS,LUG_RESIDENCIAL_EXCLUSIVO
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0,78390.0,54974.0,10.0,24.0,305.0,122,37,6
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0,58553.0,36589.0,6.0,11.0,150.0,28,42,1
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,30,3,0
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0,51431.0,39477.0,6.0,12.0,275.0,40,15,0
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0,30640.0,23725.0,4.0,15.0,242.0,38,94,1


In [23]:
#PASO 7: CARGA Y SELECCIÓN DEL DATASET COSTO_ALQUILER

# 7.1 Carga del dataset
df_alquiler = pd.read_csv(os.path.join(ruta_tablas, 'alquiler', 'COSTO_ALQUILER.csv'))

# 7.2 Visualización previa
print("-" * 70)
print("📊 Previsualización de COSTO ALQUILER:")
print("-" * 70)
display(df_alquiler.head(3))

# 7.3 Selección y Renombrado 
df_alquiler_sel = df_alquiler[['ID_ZONA', 'PRECIO_PIES_CUADRADOS_ANUAL']].copy()
df_alquiler_sel.rename(columns={'PRECIO_PIES_CUADRADOS_ANUAL': 'ALQ_PRECIO_PIE2_ANUAL'}, inplace=True)

# 7.4 Protección contra re-ejecución
if 'ALQ_PRECIO_PIE2_ANUAL' in df_master.columns:
    df_master = df_master.drop(columns=['ALQ_PRECIO_PIE2_ANUAL'])

# 7.5 Unión final al Master (Usamos left para mantener nuestras 31 zonas base)
df_master = pd.merge(df_master, df_alquiler_sel, on='ID_ZONA', how='left')

print("-" * 70)
print(f"✅ COSTO ALQUILER. Dimensiones del Master: {df_master.shape}")
print("-" * 70)
display(df_master.head())

----------------------------------------------------------------------
📊 Previsualización de COSTO ALQUILER:
----------------------------------------------------------------------


,ID_ZONA,PRECIO_PIES_CUADRADOS_ANUAL
0,MN0101,91.91
1,MN0102,122.72
2,MN0191,61.00


----------------------------------------------------------------------
✅ COSTO ALQUILER. Dimensiones del Master: (38, 22)
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE,MOV_DIA_SEMANA_PROMEDIO,MOV_FIN_DE_SEMANA_PROMEDIO,MOV_CANTIDAD_ESTACIONES,RES_COM_DIRECTA,RES_COM_INDIRECTA,LUG_COMERCIO_TURISMO,LUG_OFICINAS_ESCUELAS,LUG_RESIDENCIAL_EXCLUSIVO,ALQ_PRECIO_PIE2_ANUAL
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0,78390.0,54974.0,10.0,24.0,305.0,122,37,6,91.91
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0,58553.0,36589.0,6.0,11.0,150.0,28,42,1,122.72
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,30,3,0,61.00
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0,51431.0,39477.0,6.0,12.0,275.0,40,15,0,190.54
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0,30640.0,23725.0,4.0,15.0,242.0,38,94,1,99.03


In [27]:
#PASO 7: DATASET MAESTRO 

import os

# 1. Ruta de la carpeta de destino
ruta_final = '../../datos/maestro'
nombre_archivo = 'MASTER_DATASET_MANHATTAN.csv'
ruta_completa = os.path.join(ruta_final, nombre_archivo)

# 2. Exportamos el DataFrame 
df_master.to_csv(ruta_completa, index=False)

print("-" * 70)
print(f"📁 ARCHIVO EXPORTADO EXITOSAMENTE")
print(f"📍 Ubicación: {ruta_completa}")
print(f"📊 Estado: {df_master.shape[0]} filas y {df_master.shape[1]} columnas.")
print("-" * 70)

# 3. Visualización 
display(df_master)

----------------------------------------------------------------------
📁 ARCHIVO EXPORTADO EXITOSAMENTE
📍 Ubicación: Dataset_Final\MASTER_DATASET_MANHATTAN.csv
📊 Estado: 38 filas y 22 columnas.
----------------------------------------------------------------------


,ID_ZONA,NOMBRE_ZONA,AREA_KM2,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,SEG_DELITOS_OTROS,SEG_DELITOS_PROPIEDAD,SEG_DELITOS_TRANSPORTE,MOV_DIA_SEMANA_PROMEDIO,MOV_FIN_DE_SEMANA_PROMEDIO,MOV_CANTIDAD_ESTACIONES,RES_COM_DIRECTA,RES_COM_INDIRECTA,LUG_COMERCIO_TURISMO,LUG_OFICINAS_ESCUELAS,LUG_RESIDENCIAL_EXCLUSIVO,ALQ_PRECIO_PIE2_ANUAL
0,MN0101,Financial District-Battery Park City,19.2268,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.10,73.0,78.0,28.0,78390.0,54974.0,10.0,24.0,305.0,122,37,6,91.91
1,MN0102,Tribeca-Civic Center,13.5782,23257.0,36.8,185902.0,68.92,2.20,873.0,81.41,116.0,77.0,13.0,58553.0,36589.0,6.0,11.0,150.0,28,42,1,122.72
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,11.9933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,30,3,0,61.00
3,MN0201,SoHo-Little Italy-Hudson Square,12.9168,22167.0,39.3,132900.0,70.97,1.78,975.0,82.01,88.0,116.0,6.0,51431.0,39477.0,6.0,12.0,275.0,40,15,0,190.54
4,MN0202,Greenwich Village,10.6005,31517.0,35.7,174062.0,64.44,1.74,1003.0,78.61,93.0,80.0,16.0,30640.0,23725.0,4.0,15.0,242.0,38,94,1,99.03
5,MN0203,West Village,14.4169,32518.0,39.7,168629.0,71.17,1.68,2364.0,80.94,74.0,101.0,17.0,19220.0,16016.0,3.0,18.0,285.0,33,15,0,220.81
6,MN0301,Chinatown-Two Bridges,11.5416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141.0,42.0,3.0,8089.0,7408.0,1.0,6.0,195.0,38,20,35,61.80
7,MN0302,Lower East Side,16.4732,46107.0,42.2,58841.0,53.27,2.05,12016.0,90.54,211.0,72.0,9.0,30606.0,25371.0,3.0,14.0,218.0,51,24,79,107.90
8,MN0303,East Village,18.9848,68412.0,33.9,76136.0,60.06,1.83,11928.0,91.01,146.0,134.0,4.0,21168.0,18699.0,2.0,32.0,427.0,83,42,74,127.58
9,MN0401,Chelsea-Hudson Yards,29.6718,61878.0,39.2,126272.0,70.06,1.67,6004.0,83.90,188.0,224.0,17.0,65068.0,48740.0,6.0,24.0,354.0,69,30,26,63.64


In [26]:
#PASO 7: DATASET CONSOLIDADO ORIGINAL

import os
import pandas as pd

# 1. Aseguramos las rutas
ruta_tablas = '../../datos/base_datos'
ruta_final = '../../datos/maestro'

# 2. Definimos los archivos .csv que existen en las subcarpetas de base_datos
archivos_reales = [
    os.path.join(ruta_tablas, 'zonas', 'ZONAS.csv'),
    os.path.join(ruta_tablas, 'censo', 'CENSO.csv'),
    os.path.join(ruta_tablas, 'seguridad', 'SEGURIDAD.csv'),
    os.path.join(ruta_tablas, 'movilidad', 'MOVILIDAD.csv'),
    os.path.join(ruta_tablas, 'competencia', 'RESTAURANTES.csv'),
    os.path.join(ruta_tablas, 'lugares_interes', 'LUGARES_INTERES.csv'),
    os.path.join(ruta_tablas, 'alquiler', 'COSTO_ALQUILER.csv')
]

print("-" * 70)
print(f"📁 Archivos encontrados en datos/base_datos: {archivos_reales}")
print("-" * 70)

# 3. Usamos el primer archivo de la lista como base
if len(archivos_reales) > 0:
    # Cargamos el primero
    df_revision = pd.read_csv(archivos_reales[0])

    # 4. Unimos el resto
    for archivo in archivos_reales[1:]:
        df_temp = pd.read_csv(archivo)
        # Unimos por ID_ZONA con 'outer' para no perder nada de información bruta
        df_revision = pd.merge(df_revision, df_temp, on='ID_ZONA', how='outer')

    # 5. Exportación a la carpeta de revisión
    if not os.path.exists(ruta_final):
        os.makedirs(ruta_final)

    ruta_revision = os.path.join(ruta_final, 'DATASET_REVISION_TOTAL.csv')
    df_revision.to_csv(ruta_revision, index=False)

    print(f"✅ DATASET DE REVISIÓN GENERADO")
    print(f"📊 Columnas totales: {df_revision.shape[1]}")
    print(f"📈 Filas totales (formato largo): {df_revision.shape[0]}")
    print(f"📍 Guardado en: {ruta_revision}")
else:
    print("❌ Error: No se encontraron archivos CSV en la carpeta especificada.")

print("-" * 70)
display(df_revision.head())

----------------------------------------------------------------------
📁 Archivos encontrados en Tablas_SQL: ['CENSO.csv', 'COSTO_ALQUILER.csv', 'LUGARES_INTERES.csv', 'MOVILIDAD.csv', 'RESTAURANTES.csv', 'SEGURIDAD.csv', 'ZONAS.csv']
----------------------------------------------------------------------
✅ DATASET DE REVISIÓN GENERADO
📊 Columnas totales: 23
📈 Filas totales (formato largo): 126325
📍 Guardado en: Dataset_Final\DATASET_REVISION_TOTAL.csv
----------------------------------------------------------------------


,ID_ZONA,POBLACION_TOTAL,EDAD_MEDIANA,INGRESO_MEDIANO_HOGAR,TASA_EMPLEO,TAMANO_HOGAR_PROMEDIO,POBLACION_HISPANA,TASA_OCUPACION,PRECIO_PIES_CUADRADOS_ANUAL,CATEGORIA_LUGAR,CANTIDAD_LUGARES,TIPO_DIA,VOLUMEN_TOTAL_PASAJEROS,CANTIDAD_ESTACIONES,TIPO_COCINA,CLASIFICACION_COMPETENCIA,CANTIDAD_LOCALES,TIPO_DELITO,LUGAR_DELITO,CANTIDAD_INCIDENTES,NOMBRE_ZONA,NOMBRE_DISTRITO,AREA_KM2
0,MN0101,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.1,91.91,Administracion Publica,12,Fin de Semana,54974.0,10.0,American,Indirecta (Tráfico),160.0,ASALTO GRAVE,EDIFICIOS PUBLICOS,6.0,Financial District-Battery Park City,Manhattan,19.2268
1,MN0101,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.1,91.91,Administracion Publica,12,Fin de Semana,54974.0,10.0,American,Indirecta (Tráfico),160.0,ASALTO GRAVE,LOCAL COMERCIAL,7.0,Financial District-Battery Park City,Manhattan,19.2268
2,MN0101,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.1,91.91,Administracion Publica,12,Fin de Semana,54974.0,10.0,American,Indirecta (Tráfico),160.0,ASALTO GRAVE,OTROS,6.0,Financial District-Battery Park City,Manhattan,19.2268
3,MN0101,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.1,91.91,Administracion Publica,12,Fin de Semana,54974.0,10.0,American,Indirecta (Tráfico),160.0,ASALTO GRAVE,TRANSPORTE,18.0,Financial District-Battery Park City,Manhattan,19.2268
4,MN0101,48608.0,34.3,198961.0,79.57,1.99,1639.0,83.1,91.91,Administracion Publica,12,Fin de Semana,54974.0,10.0,American,Indirecta (Tráfico),160.0,ASALTO GRAVE,VIA PUBLICA,23.0,Financial District-Battery Park City,Manhattan,19.2268
